In [ ]:
import pandas as pd
import os
from pathlib import Path
import duckdb
from collections import defaultdict
import string

In [ ]:
CWD_DIR = os.getcwd()
RAW_DIR = os.path.join(CWD_DIR, "data\\raw")
USAGE_DIR = os.path.join(RAW_DIR, "usage-stats")

In [ ]:
def read_csv_header(file_name):
    try:
        return pd.read_csv(file_name, low_memory=False, nrows = 0).columns.tolist()
    except UnicodeDecodeError:
        return pd.read_csv(file_name, low_memory=False, nrows=0, encoding="latin-1")


In [ ]:
def read_csv_durable(file_name):
    try:
        return pd.read_csv(file_name, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(file_name, low_memory=False, encoding="latin-1")


In [ ]:
target_dir = Path(USAGE_DIR)

csv_names = [file.name for file in target_dir.iterdir() if file.suffix == ".csv"]

In [ ]:
schema_map: dict[frozenset, list[str]] = defaultdict(list)

In [ ]:
for file_name in csv_names:
    cols = read_csv_header(os.path.join(USAGE_DIR, file_name))
    schema_map[frozenset(cols)].append(file_name)

In [ ]:
lengths = {key: len(value) for key, value in schema_map.items()}

In [ ]:
SCHEMAS = [
    # v1_standard : 403 + 2 + 12 files; The 2 + 12 cases are due to unnamed columns
    {
        "Rental Id": "rental_id", "Duration": "duration_seconds", "Bike Id": "bike_id",
        "End Date": "end_date", "EndStation Id": "end_station_id", "EndStation Name": "end_station_name",
        "Start Date": "start_date", "StartStation Id": "start_station_id", "StartStation Name": "start_station_name",
    },
    # v1_logical : v1 but with logical terminals
    {
        "Rental Id": "rental_id", "Duration": "duration_seconds", "Bike Id": "bike_id",
        "End Date": "end_date", "EndStation Logical Terminal": "end_station_id", "EndStation Name": "end_station_name",
        "Start Date": "start_date", "StartStation Logical Terminal": "start_station_id", "StartStation Name": "start_station_name",
    },
    # v1_no_endstation : v1 but missing end station
    {
        "Rental Id": "rental_id", "Duration": "duration_seconds", "Bike Id": "bike_id",
        "End Date": "end_date", "EndStation Name": "end_station_name",
        "Start Date": "start_date", "StartStation Id": "start_station_id", "StartStation Name": "start_station_name",
    },
    # v2 : has spaced station names and Duration_Seconds
    {
        "Rental Id": "rental_id", "Duration_Seconds": "duration_seconds", "Bike Id": "bike_id",
        "End Date": "end_date", "End Station Id": "end_station_id", "End Station Name": "end_station_name",
        "Start Date": "start_date", "Start Station Id": "start_station_id", "Start Station Name": "start_station_name",
    },
    # v3 : identified by Number and Total duration (ms); Total duration ignored
    {
        "Number": "rental_id", "Bike number": "bike_id", "Bike model": "bike_model",
        "Start date": "start_date", "End date": "end_date",
        "Start station number": "start_station_id", "Start station": "start_station_name",
        "End station number": "end_station_id", "End station": "end_station_name",
        "Total duration (ms)": "duration_ms",
    },
]

In [ ]:
OUTPUT_COLS = [
    "rental_id","bike_id","bike_model",
    "start_date","end_date",
    "start_station_id","start_station_name",
    "end_station_id","end_station_name",
    "duration_seconds",
]

In [ ]:
#lets normalise this data


def normalise_df(df) -> pd.DataFrame:
    #identify schema
    col_set = set(df.columns)
    #because of the cases with extra leftover cols on the data, we want to use a subse
    for s in SCHEMAS:
        if set(s).issubset(col_set):
            schema = s
            break
    if schema is None:
        raise ValueError("Unrecognised Schema :(")

    cols_to_drop = [c for c in df.columns if c.startswith("Unnamed") or c == "Total Duration"]
    df = df.drop(columns = cols_to_drop)
    df = df.rename(columns=schema)[list(schema.values())]

    if "duration_ms" in df.columns:
        df["duration_seconds"] = (pd.to_numeric(df.pop("duration_ms") / 100)).round()
    
    for col in OUTPUT_COLS:
        if col not in df.columns:
            df[col] = pd.NA

    DATE_FORMAT = "%d/%m/%Y %H:%M"

    df["start_date"] = pd.to_datetime(df["start_date"], format=DATE_FORMAT, errors="coerce")
    df["end_date"]   = pd.to_datetime(df["end_date"],   format=DATE_FORMAT, errors="coerce")

    for col in ("rental_id","bike_id","start_station_id","end_station_id","duration_seconds"):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df[OUTPUT_COLS]
    


In [ ]:
test_df = pd.read_csv(os.path.join(USAGE_DIR, csv_names[370]))
test_df

In [ ]:
usage_db = duckdb.connect("usage_db.duckdb")

In [ ]:
usage_db.execute(f"""
    CREATE TABLE IF NOT EXISTS trips (
        rental_id        BIGINT,
        bike_id          BIGINT,
        bike_model       VARCHAR,
        start_date       TIMESTAMP,
        end_date         TIMESTAMP,
        start_station_id BIGINT,
        start_station_name VARCHAR,
        end_station_id   BIGINT,
        end_station_name VARCHAR,
        duration_seconds DOUBLE
    )
""")

In [ ]:
iter = 0
CHECKPOINT_EVERY = 10
for file_name in csv_names:
    iter += 1
    print(iter)
    print(file_name)
    chunk = normalise_df(read_csv_durable(os.path.join(USAGE_DIR, file_name)))
    chunk = chunk.dropna(subset=["rental_id","start_date","end_date"])
    chunk = chunk[OUTPUT_COLS]
    usage_db.execute("INSERT INTO trips SELECT * FROM chunk")
    del chunk

    if iter % CHECKPOINT_EVERY == 0:
        usage_db.execute("CHECKPOINT")
        print(f"checkpoint at file {iter}")
    